In [13]:
import pandas as pd
import numpy as np

file_path = "sample_data/Garment_Job_Risk_bulk_dataset.xlsx"
df = pd.read_excel(file_path)

df = df.dropna(axis=1, how='all')
if df.columns[0].startswith("Unnamed"):
    df = df.drop(df.columns[0], axis=1)

df.columns = [
    "Name", "Garment_Name", "Experience_Years", "Job_Role",
    "Hours_Worked", "Exposure_To_Chemicals", "Machine_Utilization", "Ergonomics",
    "Injury_History", "Health_Condition", "Safety_Training",
    "Shift_Type", "Adaptation_Frequency", "Job_Risk"
]

df.replace(["", "None", " ", "NaN", "Nan"], np.nan, inplace=True)
df.dropna(subset=["Name"], inplace=True)

df["Experience_Years"] = df["Experience_Years"].fillna(df["Experience_Years"].median())
df["Hours_Worked"] = df["Hours_Worked"].fillna(df["Hours_Worked"].median())

cat_fill_cols = ["Exposure_To_Chemicals", "Machine_Utilization", "Ergonomics", "Safety_Training", "Shift_Type"]
for col in cat_fill_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df.drop_duplicates(inplace=True)

df["Experience_Years"] = pd.to_numeric(df["Experience_Years"], errors="coerce")
df["Hours_Worked"] = pd.to_numeric(df["Hours_Worked"], errors="coerce")
df["Injury_History"] = pd.to_numeric(df["Injury_History"], errors="coerce").fillna(0).astype(int)

categorical_cols = [
    "Name", "Garment_Name", "Job_Role", "Exposure_To_Chemicals",
    "Machine_Utilization", "Ergonomics", "Health_Condition",
    "Safety_Training", "Shift_Type", "Adaptation_Frequency", "Job_Risk"
]
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

df["Garment_Name"] = df["Garment_Name"].replace({"Pallmall": "Pallmall Group"})

Q1 = df["Hours_Worked"].quantile(0.25)
Q3 = df["Hours_Worked"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df["Hours_Worked"] = df["Hours_Worked"].clip(lower_bound, upper_bound)

df = df[(df["Experience_Years"] >= 0) & (df["Experience_Years"] <= 50)]

clean_file_path = "sample_data/Cleaned_Garment_Job_Risk_FIXED.csv"
df.to_csv(clean_file_path, index=False)

print("Pipeline executed. Data aligned correctly.")
display(df.head())

Pipeline executed. Data aligned correctly.


,Name,Garment_Name,Experience_Years,Job_Role,Hours_Worked,Exposure_To_Chemicals,Machine_Utilization,Ergonomics,Injury_History,Health_Condition,Safety_Training,Shift_Type,Adaptation_Frequency,Job_Risk
0,Taslima Akter,Pallmall Group,38,Packaging,41,Medium,Light,Average,2,Healthy,Basic,Day,Occasional,Low
1,Rasel Hossain,Pallmall Group,28,Packaging,51,Low,Light,Average,0,Minor_Chronic,Basic,Day,Frequent,Low
2,Abdul Latif,Pallmall Group,14,Supervisor,68,Low,Light,Average,0,Healthy,Basic,Night,Frequent,Medium
3,Nazia Sultana,Pallmall Group,7,Sewing,28,High,Light,Average,0,Healthy,Basic,Day,Occasional,Low
4,Nazia Sultana,Pallmall Group,20,Packaging,49,Low,Light,Average,0,Healthy,Advanced,Day,Frequent,Low
